# pandas – Quick Notes

Indexing with `.loc` / `.iloc`, `groupby` aggregation variants, and boolean indexing.

## 1. Indexing: `.loc[]` vs `.iloc[]`

| Method | Selects by | Row selector | Column selector | Example |
| --- | --- | --- | --- | --- |
| **`.loc[]`** | **Labels** | Index label (`3`, `'row_a'`) | Column name (`'Name'`) | `df.loc[3, 'Name']` |
| **`.iloc[]`** | **Integer positions** (0-based) | Row position | Column position | `df.iloc[3, 1]` |
| **`df['col'][...]`** | Column, then row label | Row label | Column name | `df['Name'][3]` |

- **`.loc`**: the row label `3` and column `'Name'` → `'David'`.
- **`.iloc`**: the 4th row (position 3) and 2nd column (position 1, `'Age'`) → `40`.
- **Chained indexing** `df['Name'][3]` reads fine, but column names are case-sensitive (`df['name']` raises `KeyError`) and **assigning** through a chain (`df['Age'][3] = 41`) does not update `df` reliably — under pandas 3's Copy-on-Write it never does.

> **Rule of thumb:** use `.loc[]` when you know the labels, `.iloc[]` when you know the positions, and avoid chained `df[...][...]`, especially for writes.

⚠️ `.loc` and `.iloc` only give the same row when the index is `0, 1, 2, …`. With a custom index (e.g. `index=[10, 20, 30]`), `df.loc[1]` raises `KeyError` while `df.iloc[1]` returns the second row.

In [1]:
import pandas as pd

df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'Age': [25, 30, 35, 40, 45],
})

print(df.loc[3, 'Name'])   # label-based    → David
print(df.iloc[3, 1])       # position-based → 40
print(df['Name'][3])       # chained read   → David (avoid for writes)

David
40
David


## 2. GroupBy Sum Variants

| Variant | Work done | Returns |
| --- | --- | --- |
| `df.groupby('key')['data1'].sum()` | Aggregates only `data1` ✅ | Series |
| `df.groupby('key').sum()['data1']` | Aggregates **every** column, then keeps one ⚠️ slower | Series |
| `df[['key', 'data1']].groupby('key').sum()` | Aggregates only `data1` ✅ | DataFrame |

In [2]:
df = pd.DataFrame({
    'key': ['A', 'B', 'A', 'B', 'C'],
    'data1': [1, 2, 3, 4, 5],
    'data2': [10, 20, 30, 40, 50],
})

display(df.groupby('key')['data1'].sum())           # 1. select the column, then aggregate
display(df.groupby('key').sum()['data1'])           # 2. aggregate everything, then select
display(df[['key', 'data1']].groupby('key').sum())  # 3. select columns first → DataFrame

key
A    4
B    6
C    5
Name: data1, dtype: int64

key
A    4
B    6
C    5
Name: data1, dtype: int64

,data1
key,
A,4
B,6
C,5


## 3. 🧪 Boolean Indexing

In [3]:
df = pd.DataFrame({
    'A': [1, 5, 3, 4, 2, 6],
    'B': ['x', 'y', 'x', 'y', 'x', 'y'],
})
df

,A,B
0,1,x
1,5,y
2,3,x
3,4,y
4,2,x
5,6,y


In [4]:
# A comparison returns a boolean Series (one True/False per row)
df['A'] < 5

0     True
1    False
2     True
3     True
4     True
5    False
Name: A, dtype: bool

In [5]:
display(df[df['A'] < 5])                  # rows where A < 5
display(df[df['A'] < 5]['A'])             # just the matching A values
display(df[(df['A'] > 2) & (df['A'] < 5)])  # compound condition

print(df['A'].lt(5).sum())  # how many A < 5 → 4
print(df['A'].lt(5).any())  # is any A < 5?  → True
print(df['A'].lt(5).all())  # are all A < 5? → False

,A,B
0,1,x
2,3,x
3,4,y
4,2,x


0    1
2    3
3    4
4    2
Name: A, dtype: int64

,A,B
2,3,x
3,4,y


4
True
False


These four are equivalent ways to filter rows:

```python
df[df.A < 5]
df[df['A'] < 5]
df.loc[df['A'] < 5]
df.loc[df.A < 5]
```

To filter rows **and** pick columns, use `.loc[row_condition, columns]`:

```python
df[df.A < 5, 'B']      # ❌ error — [] does not take a (rows, columns) pair
df.loc[df.A < 5, 'B']  # ✅
```

`df.A` (attribute access) is a shortcut for `df['A']`; it only works when the column name is a valid Python identifier and doesn't clash with a DataFrame method.

In [6]:
# Combine conditions with & (and), | (or), ~ (not) — each condition in parentheses
df.loc[(df['A'] < 5) & (df['B'] == 'x'), ['A', 'B']]

,A,B
0,1,x
2,3,x
4,2,x


In [7]:
# Store the mask in a variable to reuse or inspect it
mask = df['A'] < 5
df[mask]

,A,B
0,1,x
2,3,x
3,4,y
4,2,x
